# Mymensingh Division — Urban Center Mapping (1 km)
**Complete, restartable Google Colab workflow** based on the six-factor diagram. Run cells in order. Outputs are written to Google Drive.

**Weights:** Built-up 0.30 · VIIRS 0.20 · Population 0.20 · Road 0.10 · POI 0.10 · Topography 0.10.

**Methodological cautions:** (1) This implements a **1 km** grid, not the diagram's 100 m grid. (2) Urban-center thresholds are exploratory, not validated official boundaries. (3) Road density and POI density are derived here from new source vectors for each division; their normalization must be calibrated with **shared national reference values** before interpreting scores as cross-division comparable. (4) The built-up definition is median Dynamic World `built` probability ≥0.20 during Jan–Mar 2025, aggregated to 1 km. (5) Topography uses local P5/P95 as a provisional setting; national thresholds are needed for strict comparisons.

In [ ]:
%pip -q install earthengine-api rasterio geopandas shapely scipy pandas matplotlib requests pyarrow duckdb overturemaps

In [ ]:
import os, json, time, math, subprocess, sys, warnings
import numpy as np, pandas as pd, rasterio, geopandas as gpd, ee, requests
from rasterio.transform import from_origin, Affine
from rasterio.features import rasterize, shapes
from rasterio.warp import reproject, Resampling
from scipy import ndimage
from shapely.geometry import shape, box
from google.colab import drive

drive.mount('/content/drive')
PROJECT='solid-garden-458417-t4'
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(auth_mode='colab')
    ee.Initialize(project=PROJECT)
print('Earth Engine connected:', ee.Number(2).multiply(3).getInfo())

In [ ]:
DIVISION = 'Mymensingh'
BASE=f'/content/drive/MyDrive/Urban_Center_Mapping/Mymensingh'
os.makedirs(BASE,exist_ok=True)
GRID_CRS='EPSG:32646'
GRID_M=1000
START='2025-01-01'; END='2025-04-01'
WEIGHTS=dict(built=0.30,viirs=0.20,population=0.20,road=0.10,poi=0.10,topography=0.10)
# Khulna national reference values from earlier analysis (not independent national verification)
VIIRS_NATIONAL_P99=7.463509998613131
POP_NATIONAL_P99=17142.65411779742
# Set identical national constants for every division once measured.
ROAD_SHARED_P99=None
POI_SHARED_P99=None
TOPO_ELEV_SHARED_P5_P95=None
TOPO_SLOPE_SHARED_P5_P95=None
SCORE_THRESHOLD=0.55
MIN_PATCH_KM2=1.0
MIN_MEAN_SCORE=0.60
MIN_MEAN_POP_DENSITY=500.0
print('Output folder:',BASE)

In [ ]:
# GAUL 2015 Level 1 boundary: inspect exact names before selecting.
ga=ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.eq('ADM0_NAME','Bangladesh'))
names=ga.aggregate_array('ADM1_NAME').distinct().getInfo()
print('Available divisions:',names)
matches=[x for x in names if x.lower().replace(' ','')==DIVISION.lower().replace(' ','')]
if not matches:
    raise ValueError(f'Division {DIVISION} not found. Choose one of {names}')
roi=ga.filter(ee.Filter.eq('ADM1_NAME',matches[0])).geometry()
roi_json=roi.getInfo()
from shapely.geometry import shape as shapely_shape
roi_wgs=shapely_shape(roi_json)
roi_gdf=gpd.GeoDataFrame(geometry=[roi_wgs],crs='EPSG:4326').to_crs(GRID_CRS)
geom=roi_gdf.geometry.iloc[0]
minx,miny,maxx,maxy=geom.bounds
left=math.floor(minx/GRID_M)*GRID_M; right=math.ceil(maxx/GRID_M)*GRID_M
top=math.ceil(maxy/GRID_M)*GRID_M; bottom=math.floor(miny/GRID_M)*GRID_M
WIDTH=int(round((right-left)/GRID_M)); HEIGHT=int(round((top-bottom)/GRID_M))
TRANSFORM=from_origin(left,top,GRID_M,GRID_M)
assert WIDTH*HEIGHT < 500000
region=ee.Geometry.Rectangle([left,bottom,right,top],proj=GRID_CRS,geodesic=False)
inside=rasterize([(geom,1)],out_shape=(HEIGHT,WIDTH),transform=TRANSFORM,fill=0,dtype='uint8').astype(bool)
print('Grid:',HEIGHT,WIDTH,'inside:',inside.sum(),'CRS:',GRID_CRS)
roi_gdf.to_file(os.path.join(BASE,f'{DIVISION}_Boundary.gpkg'),driver='GPKG')

In [ ]:
def save_tif(name,arr,nodata=-9999):
    path=os.path.join(BASE,f'{DIVISION}_{name}_1km.tif')
    a=np.asarray(arr,dtype='float32')
    with rasterio.open(path,'w',driver='GTiff',width=WIDTH,height=HEIGHT,count=1,
                       dtype='float32',crs=GRID_CRS,transform=TRANSFORM,nodata=nodata,
                       compress='lzw') as dst:
        dst.write(np.where(np.isfinite(a)&inside,a,nodata).astype('float32'),1)
    print('Saved:',path)
    return path

def read_tif(name):
    path=os.path.join(BASE,f'{DIVISION}_{name}_1km.tif')
    with rasterio.open(path) as src:
        if src.shape!=(HEIGHT,WIDTH) or src.crs.to_string()!=GRID_CRS or not src.transform.almost_equals(TRANSFORM):
            raise ValueError(f'Grid mismatch: {path}')
        return src.read(1,masked=True).filled(np.nan).astype('float32')

def ee_download_tiles(image,name,tile_cells=10):
    # Small direct requests avoid the full-area 10m reprojection limit.
    path=os.path.join(BASE,f'{DIVISION}_{name}_1km.tif')
    if os.path.exists(path):
        print('Reusing:',path)
        return read_tif(name)
    progress=os.path.join(BASE,f'{DIVISION}_{name}_progress.npz')
    nr=math.ceil(HEIGHT/tile_cells);nc=math.ceil(WIDTH/tile_cells)
    if os.path.exists(progress):
        with np.load(progress) as z:
            result=z['result'].copy(); done=z['done'].copy()
        if result.shape!=(HEIGHT,WIDTH) or done.shape!=(nr,nc):
            raise ValueError('Progress grid mismatch')
    else:
        result=np.full((HEIGHT,WIDTH),np.nan,dtype='float32');done=np.zeros((nr,nc),bool)
    errors=[]
    for ir in range(nr):
        for ic in range(nc):
            if done[ir,ic]:continue
            r0=ir*tile_cells;c0=ic*tile_cells
            r1=min(r0+tile_cells,HEIGHT);c1=min(c0+tile_cells,WIDTH)
            if not inside[r0:r1,c0:c1].any():
                done[ir,ic]=True;continue
            x0=left+c0*GRID_M;x1=left+c1*GRID_M
            y1=top-r0*GRID_M;y0=top-r1*GRID_M
            tile_geom=ee.Geometry.Rectangle([x0,y0,x1,y1],proj=GRID_CRS,geodesic=False)
            tr=[GRID_M,0,x0,0,-GRID_M,y1]
            err=''
            for attempt in range(3):
                try:
                    url=image.getDownloadURL({'region':tile_geom,'crs':GRID_CRS,
                                               'crs_transform':tr,'format':'GEO_TIFF'})
                    resp=requests.get(url,timeout=180)
                    if resp.status_code!=200:raise RuntimeError(resp.text[:500])
                    with rasterio.MemoryFile(resp.content) as mem:
                        with mem.open() as src:
                            tile=src.read(1,masked=True).filled(np.nan)
                            if tile.shape!=(r1-r0,c1-c0) or not src.transform.almost_equals(Affine(*tr)):
                                raise ValueError('Tile shape/transform mismatch')
                    result[r0:r1,c0:c1]=tile;done[ir,ic]=True;break
                except Exception as exc:
                    err=str(exc);time.sleep(2*(attempt+1))
            if not done[ir,ic]:errors.append((ir,ic,err))
        np.savez_compressed(progress,result=result,done=done)
        print(name,'tile rows',ir+1,'/',nr,'complete:',int(done.sum()),'/',done.size)
    if errors:
        print('First failures:',errors[:5]);raise RuntimeError(f'{len(errors)} tile(s) failed. Rerun to resume.')
    save_tif(name,result)
    return np.where(inside,result,np.nan)

## 1. Built-up fraction — Dynamic World
Compute median `built` probability, threshold ≥0.20, then average 10 m binary cells within each 1 km cell. Direct download in small tiles; progress saved to Drive. Note that mosaicked native projections can introduce seam effects; inspect the output before publication.

In [ ]:
dw=ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filterBounds(roi).filterDate(START,END).select('built')
print('Dynamic World scenes:',dw.size().getInfo())
if dw.size().getInfo()==0:raise RuntimeError('No Dynamic World images')
native=ee.Image(dw.first()).select('built').projection()
binary=dw.median().gte(0.20).toFloat().setDefaultProjection(native)
built_ee=binary.reduceResolution(reducer=ee.Reducer.mean(),maxPixels=16384).rename('built_fraction')
built=ee_download_tiles(built_ee,'Builtup_Fraction',tile_cells=10)
print('Built-up fraction:',np.nanmin(built),np.nanmean(built),np.nanmax(built))

## 2. VIIRS and GHSL population
VIIRS Jan–Mar 2025 and GHSL 2025 are normalized using **the same previously adopted national P99 constants** for each division. Population absolute density is also saved for the ≥500 people/km² patch filter. Check GHSL band availability in the console.

In [ ]:
viirs_col=(ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
           .filterBounds(roi).filterDate(START,END).select('avg_rad'))
print('VIIRS months:',viirs_col.size().getInfo())
viirs_raw=viirs_col.mean().max(0)
viirs_ee=viirs_raw.divide(VIIRS_NATIONAL_P99).clamp(0,1).rename('viirs')
viirs=ee_download_tiles(viirs_ee,'VIIRS_Score',tile_cells=20)
# GHSL population grid: people per cell, not people/km2. Convert using source nominal pixel area.
ghsl=ee.Image('JRC/GHSL/P2023A/GHS_POP/2025').select('population_count')
print('GHSL bands:',ghsl.bandNames().getInfo())
# Convert native GHSL population count to density using the native pixel area.
# At 1 km, use sum of source counts divided by 1 km2; avoid resampling a count with nearest neighbor.
pop_count=ghsl.setDefaultProjection(ghsl.projection())
pop_1km=pop_count.reduceResolution(ee.Reducer.sum(),maxPixels=4096)
pop_density=ee_download_tiles(pop_1km,'Population_Density_PeopleKm2',tile_cells=20)
pop=np.clip(pop_density/POP_NATIONAL_P99,0,1)
save_tif('Population_Score',pop)

## 3. Road density — GRIP4
Download GRIP4 road features intersecting the division as GeoJSON in manageable spatial chunks. **This is potentially slow** and requires Google Earth Engine permissions. Road length is measured in projected metres per 1 km² cell. Uses `geopandas.overlay`/intersection, not an uncalibrated binary road-presence raster.

In [ ]:
# GRIP4 Southeast Asia, consistent with the earlier Khulna analysis.
roads_ee=ee.FeatureCollection('projects/sat-io/open-datasets/GRIP4/South-East-Asia').filterBounds(roi)
print('GRIP4 features intersecting division:',roads_ee.size().getInfo())
# To avoid large EE JSON requests, export one vector file to Google Drive (once).
ROAD_VECTOR=os.path.join(BASE,f'{DIVISION}_GRIP4_roads.geojson')
if not os.path.exists(ROAD_VECTOR):
    print('Road vector not present. Exporting via geemap in batches can be slow.')
    import geemap
    # A Drive export is safer for large vector collections than getInfo() over thousands of roads.
    task=ee.batch.Export.table.toDrive(collection=roads_ee,description=f'{DIVISION}_GRIP4_roads',
                                       folder='Urban_Center_Mapping',fileNamePrefix=f'{DIVISION}_GRIP4_roads',fileFormat='GeoJSON')
    task.start()
    print('Started road export:',task.id)
    raise RuntimeError('Wait until the road GeoJSON export finishes, then copy it from Drive Urban_Center_Mapping into '+ROAD_VECTOR+' and rerun this cell.')
roads=gpd.read_file(ROAD_VECTOR).to_crs(GRID_CRS)
roads=roads[roads.geometry.notna() & ~roads.geometry.is_empty]
roads=roads.clip(roi_gdf)
print('Road features loaded:',len(roads))

In [ ]:
# Exact road length per cell using spatial index, with checkpoint output.
ROAD_RASTER=os.path.join(BASE,f'{DIVISION}_Road_Density_1km.tif')
if os.path.exists(ROAD_RASTER):
    road=read_tif('Road_Density')
else:
    road=np.full((HEIGHT,WIDTH),np.nan,dtype='float32')
    lines=roads[roads.geometry.geom_type.isin(['LineString','MultiLineString'])].copy()
    if lines.empty:raise RuntimeError('No line geometries found in GRIP4')
    sidx=lines.sindex
    for row in range(HEIGHT):
        for col in np.flatnonzero(inside[row]):
            cell=box(left+col*GRID_M,top-(row+1)*GRID_M,left+(col+1)*GRID_M,top-row*GRID_M)
            cand=lines.iloc[list(sidx.intersection(cell.bounds))]
            road[row,col]=sum(g.intersection(cell).length for g in cand.geometry if g.intersects(cell))/1000.0
        if row%25==0:print('Road row',row,'/',HEIGHT)
    save_tif('Road_Density',road)
# Shared national reference is required for true inter-division comparisons.
road_p99=ROAD_SHARED_P99 or float(np.nanpercentile(road[inside],99))
road_score=np.clip(road/road_p99,0,1)
save_tif('Road_Score',road_score)
print('Road P99 used:',road_p99,'km/km2; nationally comparable:',ROAD_SHARED_P99 is not None)

## 4. Overture POIs / services
Use Overture Maps **places** GeoParquet downloaded for this division's bounding box. A single 1 km cell receives POI counts within a **2.1 km square** (same window type as the earlier Khulna analysis). This is *not* POI/km² density. Overture release and category filters must be held fixed across divisions. The Overture command can require network access and its schema/release may change; if download fails, fix the source rather than substituting arbitrary points.

In [ ]:
from scipy.ndimage import uniform_filter
POI_FILE=os.path.join(BASE,f'{DIVISION}_Overture_Places.parquet')
if not os.path.exists(POI_FILE):
    # Use CLI to fetch only division bbox. Output may be large and include points outside boundary.
    lon0,lat0,lon1,lat1=roi_wgs.bounds
    cmd=[sys.executable,'-m','overturemaps','download','--bbox',f'{lon0},{lat0},{lon1},{lat1}',
         '-f','geoparquet','--type','place','-o',POI_FILE]
    print('Downloading Overture places; this may take time.')
    print(' '.join(cmd))
    subprocess.run(cmd,check=True)
poi_gdf=gpd.read_parquet(POI_FILE)
print('Overture columns:',list(poi_gdf.columns))
poi_gdf=poi_gdf.to_crs(GRID_CRS)
poi_gdf=poi_gdf[poi_gdf.geometry.within(geom)]
# Category selection: match against broad urban-services vocabulary; inspect counts.
URBAN_TERMS=('shopping','store','market','mall','school','university','college','hospital',
             'clinic','doctor','pharmacy','restaurant','cafe','bank','hotel','office',
             'commercial','service','government','library','bus_station','train_station')
def category_text(x):
    if isinstance(x,dict):return str(x.get('primary',''))
    return str(x)
if 'categories' not in poi_gdf.columns:
    raise RuntimeError('Overture places categories field missing; inspect schema before continuing')
poi_gdf=poi_gdf[poi_gdf['categories'].map(lambda x:any(k in category_text(x).lower() for k in URBAN_TERMS))]
print('Selected urban POIs:',len(poi_gdf))
poi_count=rasterize([(g,1) for g in poi_gdf.geometry if g is not None],out_shape=(HEIGHT,WIDTH),
                    transform=TRANSFORM,fill=0,merge_alg=rasterio.enums.MergeAlg.add,dtype='int32')
# 2.1 km square kernel: at 1 km, 3x3 is an approximation; original Khulna
# raster used a moving-window statistic that may not be numerically identical.
poi_concentration=uniform_filter(poi_count.astype('float32'),size=3,mode='constant')*9
poi_concentration[~inside]=np.nan
save_tif('POI_Concentration',poi_concentration)
poi_p99=POI_SHARED_P99 or float(np.nanpercentile(poi_concentration[inside],99))
poi_score=np.clip(poi_concentration/max(poi_p99,1e-6),0,1)
save_tif('POI_Score',poi_score)
print('POI P99:',poi_p99,'Nationally comparable:',POI_SHARED_P99 is not None)

## 5. Topography — Copernicus GLO30
Slope is calculated on individual native DEM tiles, then mosaicked. Elevation and slope normalized inversely P5–P95. Score = 0.40 elevation + 0.60 slope. The **low elevation = suitability** assumption is questionable in flood-prone regions; review before planning use.

In [ ]:
dem_col=ee.ImageCollection('COPERNICUS/DEM/GLO30').filterBounds(roi).select('DEM')
print('DEM tiles:',dem_col.size().getInfo())
# Compute slope on each original DEM tile, not the unprojected mosaic.
elev=dem_col.mosaic().rename('elevation')
slope=dem_col.map(lambda im:ee.Terrain.slope(im).rename('slope')).mosaic()
elev_arr=ee_download_tiles(elev,'Elevation_M',tile_cells=20)
slope_arr=ee_download_tiles(slope,'Slope_Deg',tile_cells=20)
def inverse_percentile(a,shared=None):
    vals=a[inside & np.isfinite(a)]
    p5,p95=shared if shared is not None else np.percentile(vals,[5,95])
    if p95<=p5:raise ValueError('Invalid topography percentile range')
    return np.clip((p95-a)/(p95-p5),0,1)
topography=0.4*inverse_percentile(elev_arr,TOPO_ELEV_SHARED_P5_P95)+0.6*inverse_percentile(slope_arr,TOPO_SLOPE_SHARED_P5_P95)
save_tif('Topography_Score',topography)

## 6. Six-factor score, candidate mask, patch statistics, polygons
**Strict missing-data rule:** any missing indicator gives NoData, not zero. Filtering uses area ≥1 km², mean score >0.60, and **actual** mean population density >500 people/km².

In [ ]:
factors={'built':built,'viirs':viirs,'population':pop,'road':road_score,'poi':poi_score,'topography':topography}
valid=inside.copy()
for key,a in factors.items():
    if a.shape!=(HEIGHT,WIDTH):raise ValueError(key+' grid mismatch')
    valid &= np.isfinite(a)&(a>=0)&(a<=1)
score=np.full((HEIGHT,WIDTH),np.nan,dtype='float32')
weighted=sum(WEIGHTS[k]*np.nan_to_num(v,nan=0) for k,v in factors.items())
score[valid]=weighted[valid]
save_tif('Urban_Score_6Factor',score)
print('Valid score pixels:',valid.sum(),'Range:',np.nanmin(score),np.nanmax(score))
if ROAD_SHARED_P99 is None or POI_SHARED_P99 is None or TOPO_ELEV_SHARED_P5_P95 is None or TOPO_SLOPE_SHARED_P5_P95 is None:
    print('WARNING: Some factors use division-specific normalization; cross-division scores are not yet strictly comparable.')

In [ ]:
candidate=valid&(score>SCORE_THRESHOLD)
labels,n=ndimage.label(candidate,structure=np.ones((3,3),dtype=int))
rows=[];keep=np.zeros((HEIGHT,WIDTH),bool)
for label in range(1,n+1):
    cells=labels==label
    area=float(cells.sum()) # each 1 km pixel is 1 km2
    mean_score=float(np.nanmean(score[cells]))
    mean_pop=float(np.nanmean(pop_density[cells]))
    accepted=(area>=MIN_PATCH_KM2 and mean_score>MIN_MEAN_SCORE and mean_pop>MIN_MEAN_POP_DENSITY)
    rows.append(dict(patch_id=label,area_km2=area,mean_score=mean_score,
                     mean_pop_density=mean_pop,accepted=accepted))
    if accepted:keep|=cells
summary=pd.DataFrame(rows,columns=['patch_id','area_km2','mean_score','mean_pop_density','accepted'])
summary.to_csv(os.path.join(BASE,f'{DIVISION}_Patch_Statistics.csv'),index=False)
save_tif('Urban_Center_Candidate_Mask',candidate.astype('float32'))
save_tif('Urban_Center_Final_Mask',keep.astype('float32'))
print('Candidate patches:',n,'Accepted patches:',sum(x['accepted'] for x in rows))

In [ ]:
from shapely.geometry import shape as to_shape
records=[]
for g,v in shapes(labels.astype('int32'),mask=keep,transform=TRANSFORM):
    if v==0:continue
    poly=to_shape(g)
    s=summary[summary.patch_id==int(v)]
    if s.empty:continue
    r=s.iloc[0]
    records.append(dict(geometry=poly,patch_id=int(v),area_km2=float(r.area_km2),
                        mean_score=float(r.mean_score),mean_pop=float(r.mean_pop_density)))
cols=['patch_id','area_km2','mean_score','mean_pop','geometry']
polys=gpd.GeoDataFrame(records,columns=cols,geometry='geometry',crs=GRID_CRS)
poly_path=os.path.join(BASE,f'{DIVISION}_Urban_Centers_Final.gpkg')
if len(polys):polys.to_file(poly_path,driver='GPKG')
else:print('No polygons meet the current thresholds; no empty GPKG written.')
print('Final polygon count:',len(polys))

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(9,10))
im=ax.imshow(score,cmap='viridis',vmin=0,vmax=1)
if len(polys):polys.boundary.plot(ax=ax,edgecolor='red',linewidth=1)
ax.set_title(f'{DIVISION} — Six-Factor Urban Score (1 km)')
ax.axis('off');fig.colorbar(im,ax=ax,shrink=.65,label='Urban score (0–1)')
fig.savefig(os.path.join(BASE,f'{DIVISION}_Urban_Score_Map.png'),dpi=200,bbox_inches='tight')
plt.show()
metadata={'division':DIVISION,'grid_m':GRID_M,'period':[START,END],
          'weights':WEIGHTS,'threshold':SCORE_THRESHOLD,'min_patch_km2':MIN_PATCH_KM2,
          'min_mean_score':MIN_MEAN_SCORE,'min_mean_pop_density':MIN_MEAN_POP_DENSITY,
          'national_comparability_confirmed':all(x is not None for x in [ROAD_SHARED_P99,POI_SHARED_P99,TOPO_ELEV_SHARED_P5_P95,TOPO_SLOPE_SHARED_P5_P95]),
          'poi_window':'3x3 cells approximation to 2.1 km square; not exactly identical to earlier Khulna moving window',
          'data_notes':'GRIP4, Overture Maps, Dynamic World, VIIRS, GHSL, Copernicus DEM'}
with open(os.path.join(BASE,f'{DIVISION}_Methodology.json'),'w') as f:json.dump(metadata,f,indent=2)
print('Done. All outputs:',BASE)